# CDS550 — RoBERTa-large fine-tuning on Colab

**运行顺序**

1. `代码执行程序` → `更改运行时类型` → 选 GPU（A100 / G4；L4 不比本地 5070 Ti 快）
2. 跑第 2 格确认显卡型号
3. 跑第 3 格安装依赖
4. 挂载 Drive（第 4 格）
5. 在第 5 格填 `RAW_BASE`，然后运行 —— 自动下载并解压数据
6. 往下依次运行

**为什么用 zip**：两个 CSV 各 32MB，超过 GitHub 网页上传的 25MB 单文件上限；
压成 `train.zip` / `test.zip` 各约 13MB，网页就能传。

**断点续跑**：每折结束立刻把预测存进 Google Drive。断线后重跑训练格，
已完成的折会自动跳过（检测 `fold{k}.npz` 是否存在）。

A100 上 5 折约 3 小时；G4 约 2 小时。

In [ ]:
import torch, subprocess
print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout)
print('torch', torch.__version__, '| cuda', torch.cuda.is_available())
if torch.cuda.is_available():
    print('device    :', torch.cuda.get_device_name(0))
    print('capability:', torch.cuda.get_device_capability(0))
    print('vram GB   : %.1f' % (torch.cuda.get_device_properties(0).total_memory / 1e9))
else:
    raise SystemExit('No GPU. Runtime -> Change runtime type -> GPU')

In [ ]:
!pip install -q transformers scikit-learn
import transformers, sklearn
print('transformers', transformers.__version__, '| sklearn', sklearn.__version__)

### 数据获取（四选一）

**A. GitHub（推荐，手机端最省事）**

仓库里放 `train.zip`、`test.zip`（和这个 notebook），下一格填：

```
https://raw.githubusercontent.com/romantic-xrq/cds550/main
```

程序会先试 `.zip` 再退回 `.csv`，两种都认。

**B. Google Drive**

把两个 CSV（或 zip）传到 `MyDrive/cds550_data/`，下一格把 `DRIVE_DIR` 指过去。

**C. Kaggle API**

`kaggle.com` → 头像 → Settings → API → Create New Token，填 `username` / `key`。

**D. 手动上传**

三个变量都留空，用左侧文件面板把 CSV 拖进 `/content/`。

In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')
os.makedirs('/content/drive/MyDrive/cds550', exist_ok=True)
print('drive ready')

In [ ]:
import os, glob, json, shutil

RAW_BASE        = ''   # A: https://raw.githubusercontent.com/romantic-xrq/cds550/main
DRIVE_DIR       = ''   # B: /content/drive/MyDrive/cds550_data
KAGGLE_USERNAME = ''   # C
KAGGLE_KEY      = ''   # C
SLUG            = ''   # C: cds-550-individual-project-movie-review-sentiment-analysis

def fetch(url, path):
    rc = os.system('wget -q --timeout=60 "%s" -O "%s"' % (url, path))
    return rc == 0 and os.path.exists(path) and os.path.getsize(path) > 1024

if RAW_BASE:
    for f in ('train', 'test'):
        zp = '/content/%s.zip' % f
        if fetch('%s/%s.zip' % (RAW_BASE, f), zp):
            os.system('unzip -o -q "%s" -d /content/' % zp)
            print('%s.zip -> ok' % f)
        elif fetch('%s/%s.csv' % (RAW_BASE, f), '/content/%s.csv' % f):
            print('%s.csv -> ok' % f)
        else:
            print('%s -> FAILED (check RAW_BASE / repo visibility)' % f)

elif DRIVE_DIR:
    for f in ('train', 'test'):
        for pat in (DRIVE_DIR + '/**/' + f + '.zip', DRIVE_DIR + '/**/' + f + '.csv'):
            hit = glob.glob(pat, recursive=True)
            if hit:
                if hit[0].endswith('.zip'):
                    os.system('unzip -o -q "%s" -d /content/' % hit[0])
                else:
                    shutil.copy(hit[0], '/content/%s.csv' % f)
                break

elif KAGGLE_USERNAME and KAGGLE_KEY:
    os.makedirs('/root/.kaggle', exist_ok=True)
    p = '/root/.kaggle/kaggle.json'
    json.dump({'username': KAGGLE_USERNAME, 'key': KAGGLE_KEY}, open(p, 'w'))
    os.chmod(p, 0o600)
    os.makedirs('/content/data', exist_ok=True)
    os.system('pip install -q kaggle')
    os.system('kaggle competitions download -c %s -p /content/data' % SLUG)
    os.system('unzip -o -q /content/data/*.zip -d /content/data')
    for f in ('train.csv', 'test.csv'):
        hit = glob.glob('/content/data/**/' + f, recursive=True)
        if hit:
            shutil.copy(hit[0], '/content/' + f)

else:
    print('no remote source configured -> assuming manual upload')

missing = [f for f in ('/content/train.csv', '/content/test.csv') if not os.path.exists(f)]
print('MISSING:', missing if missing else 'none')
if not missing:
    print(os.popen('ls -lh /content/train.csv /content/test.csv').read())
    import pandas as pd
    print('train cols:', pd.read_csv('/content/train.csv', nrows=3).columns.tolist())

In [ ]:
import os, json, time, html, re
import numpy as np, pandas as pd, torch
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from torch.utils.data import Dataset, DataLoader
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          get_linear_schedule_with_warmup)
import transformers
transformers.logging.set_verbosity_error()

CFG = dict(model='roberta-large', folds=5, epochs=3, max_len=512,
           batch_size=8, eval_batch_size=32, lr=1e-5, weight_decay=0.01,
           warmup_ratio=0.1, max_grad_norm=1.0, seed=42)

OUT = '/content/drive/MyDrive/cds550'
os.makedirs(OUT, exist_ok=True)
print('outputs ->', OUT)
print('A100 显存充裕，可把 batch_size 调到 16 或 32 再快 20-30%')

In [ ]:
_TAG = re.compile(r'</?[a-zA-Z][^>]{0,30}>')
_WS  = re.compile(r'\s+')

def clean_text(t):
    if not isinstance(t, str):
        return ''
    t = html.unescape(t)
    t = _TAG.sub(' ', t)
    return _WS.sub(' ', t).strip()

train = pd.read_csv('/content/train.csv')
test  = pd.read_csv('/content/test.csv')
train['text'] = train['review'].map(clean_text)
test['text']  = test['review'].map(clean_text)
train['y'] = train['sentiment'].map({'negative': 0, 'positive': 1})
print('train', train.shape, '| test', test.shape)
print(train.sentiment.value_counts().to_dict())

class DS(Dataset):
    def __init__(self, enc, labels=None):
        self.enc, self.labels = enc, labels
    def __len__(self):
        return len(self.enc['input_ids'])
    def __getitem__(self, i):
        it = {k: torch.tensor(v[i]) for k, v in self.enc.items()}
        if self.labels is not None:
            it['labels'] = torch.tensor(int(self.labels[i]))
        return it

def collate(tok):
    def f(batch):
        feats = [{k: v for k, v in b.items() if k != 'labels'} for b in batch]
        enc = tok.pad(feats, padding=True, return_tensors='pt')
        if 'labels' in batch[0]:
            enc['labels'] = torch.stack([b['labels'] for b in batch])
        return enc
    return f

In [ ]:
@torch.no_grad()
def predict(model, loader, device):
    model.eval()
    out = []
    for batch in loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        with torch.autocast('cuda', dtype=torch.bfloat16):
            logits = model(**batch).logits.float()
        out.append(torch.softmax(logits, -1)[:, 1].cpu().numpy())
    return np.concatenate(out)

def train_fold(tr_x, tr_y, va_x, va_y, te_x, tok, device, tag):
    m = AutoModelForSequenceClassification.from_pretrained(CFG['model'], num_labels=2).to(device)
    col = collate(tok)
    tr_ld = DataLoader(DS(tok(tr_x, truncation=True, max_length=CFG['max_len']), tr_y),
                       batch_size=CFG['batch_size'], shuffle=True, collate_fn=col)
    va_ld = DataLoader(DS(tok(va_x, truncation=True, max_length=CFG['max_len'])),
                       batch_size=CFG['eval_batch_size'], collate_fn=col)
    te_ld = DataLoader(DS(tok(te_x, truncation=True, max_length=CFG['max_len'])),
                       batch_size=CFG['eval_batch_size'], collate_fn=col)

    opt = torch.optim.AdamW(m.parameters(), lr=CFG['lr'], weight_decay=CFG['weight_decay'])
    total = len(tr_ld) * CFG['epochs']
    sch = get_linear_schedule_with_warmup(opt, int(total * CFG['warmup_ratio']), total)

    best_acc, best_state = -1.0, None
    for ep in range(1, CFG['epochs'] + 1):
        m.train(); t0 = time.time()
        for step, batch in enumerate(tr_ld, 1):
            batch = {k: v.to(device) for k, v in batch.items()}
            opt.zero_grad(set_to_none=True)
            with torch.autocast('cuda', dtype=torch.bfloat16):
                loss = m(**batch).loss
            loss.backward()
            torch.nn.utils.clip_grad_norm_(m.parameters(), CFG['max_grad_norm'])
            opt.step(); sch.step()
            if step % 200 == 0:
                print('  %s ep%d %d/%d loss=%.4f (%.0fs)' % (tag, ep, step, len(tr_ld), loss.item(), time.time()-t0), flush=True)
        acc = accuracy_score(va_y, (predict(m, va_ld, device) >= .5).astype(int))
        print('  %s ep%d val_acc=%.4f' % (tag, ep, acc), flush=True)
        if acc > best_acc:
            best_acc = acc
            best_state = {k: v.detach().clone() for k, v in m.state_dict().items()}

    m.load_state_dict(best_state)
    va_p = predict(m, va_ld, device)
    te_p = predict(m, te_ld, device)
    del m
    torch.cuda.empty_cache()
    return va_p, te_p, best_acc

In [ ]:
torch.manual_seed(CFG['seed']); np.random.seed(CFG['seed'])
device = torch.device('cuda')
tok = AutoTokenizer.from_pretrained(CFG['model'])

X = train['text'].to_numpy(); y = train['y'].to_numpy()
te_x = test['text'].tolist()
skf = StratifiedKFold(n_splits=CFG['folds'], shuffle=True, random_state=CFG['seed'])
splits = list(skf.split(X, y))

for k, (tr_idx, va_idx) in enumerate(splits, 1):
    fpath = '%s/fold%d.npz' % (OUT, k)
    if os.path.exists(fpath):
        print('fold %d already done, skipping' % k); continue
    print('\n=== fold %d/%d ===' % (k, CFG['folds']), flush=True)
    va_p, te_p, acc = train_fold(X[tr_idx].tolist(), y[tr_idx], X[va_idx].tolist(),
                                 y[va_idx], te_x, tok, device, 'f%d' % k)
    np.savez_compressed(fpath, va_idx=va_idx, va_prob=va_p, te_prob=te_p, acc=acc)
    print('fold %d saved -> %s  (acc %.4f)' % (k, fpath, acc))

print('\nall folds finished')

In [ ]:
oof = np.zeros(len(train)); te_prob = np.zeros(len(test)); accs = []
for k, (tr_idx, va_idx) in enumerate(splits, 1):
    d = np.load('%s/fold%d.npz' % (OUT, k))
    oof[d['va_idx']] = d['va_prob']
    te_prob += d['te_prob'] / CFG['folds']
    accs.append(float(d['acc']))

pred = (oof >= .5).astype(int)
res = dict(model=CFG['model'], folds=CFG['folds'], epochs=CFG['epochs'],
           max_len=CFG['max_len'], lr=CFG['lr'],
           cv_mean=float(np.mean(accs)), cv_std=float(np.std(accs)),
           oof_accuracy=float(accuracy_score(y, pred)),
           oof_precision=float(precision_score(y, pred)),
           oof_recall=float(recall_score(y, pred)),
           oof_f1=float(f1_score(y, pred)))
print(json.dumps(res, indent=2))

np.savez_compressed('%s/transformer_probs_roberta_large.npz' % OUT, oof=oof, test=te_prob)
json.dump(res, open('%s/transformer_results_roberta_large.json' % OUT, 'w'), indent=2)

test_pred = (te_prob >= 0.5).astype(int)
sub = pd.DataFrame({'id': test['id'],
                    'sentiment': np.where(test_pred == 1, 'positive', 'negative')})
sub.to_csv('%s/submission_roberta_large.csv' % OUT, index=False)
print('rows', len(sub), '| label dist', sub.sentiment.value_counts().to_dict())
print('saved to', OUT)

In [ ]:
from google.colab import files
for f in ('transformer_probs_roberta_large.npz',
          'transformer_results_roberta_large.json',
          'submission_roberta_large.csv'):
    files.download('%s/%s' % (OUT, f))